In [0]:
# --------------------------------------------------------------------------- 
# SCRIPT: 1.5_evolucao_tematica_frentes
# LOCAL: 3_gold/src/1_atlas_frentes/
# OBJETIVO: Evolução temática das frentes (dependência as tabelas geradas pelos scripts 
#           2_Silver\1_atlas_frentes\1.2_frentes_transform e 1.5_frentes_historico_ingest)
# ENTREGA: Evolução do número de frentes por tema entre legislaturas
# --------------------------------------------------------------------------- 

from pyspark.sql.functions import col, when, lower, lit

# Carregar frentes atuais (57) e anteriores (56)
df_57 = spark.table("silver_frentes").withColumn("legislatura", lit("57"))
df_56 = spark.table("bronze_frentes_legislat_56").select("id", "idLegislatura", "titulo") \
             .withColumnRenamed("idLegislatura", "idLegislatura").withColumn("legislatura", lit("56"))

# Unir as duas bases
df_unificado = df_57.unionByName(df_56)

# Classificar por Temas
df_tematico = df_unificado.withColumn("tema", 
    when(lower(col("titulo")).contains("nanismo"), "Nanimso")
    .when(lower(col("titulo")).contains("ferrovi"), "Ferroviária")
    .when(lower(col("titulo")).contains("telecom"), "Soluções Digitais")
    .when(lower(col("titulo")).contains("universidades"), "Universidades")
    .when(lower(col("titulo")).contains("educação"), "Educação")
    .when(lower(col("titulo")).contains("saúde"), "Saúde")
    .when(lower(col("titulo")).contains("infraestrutura"), "Infra")
    .when(lower(col("titulo")).contains("agro"), "Agro")
    .when(lower(col("titulo")).contains("ambiental"), "Ambiental")
    .when(lower(col("titulo")).contains("cultura"), "Cultura")
    .otherwise("z- Outros")
)

# Agrupar para ver a evolução
df_evolucao = df_tematico.groupBy("tema", "legislatura").count().orderBy("tema", "legislatura") 

# Salvar na Gold
df_evolucao.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_atlas_evolucao_frentes")

display(df_evolucao)